In [1]:
# Loading Cleaned Data 

In [1]:
import pandas as pd 
import numpy as np

In [2]:
tickets = pd.read_csv(r'C:\Users\bodhe\Downloads\DS\____Projects_____\ticket_pricing_project\data\cleaned\cleaned_ticket_sales.csv')
weather = pd.read_csv(r'C:\Users\bodhe\Downloads\DS\____Projects_____\ticket_pricing_project\data\cleaned\cleaned_weather_data.csv')
competitors = pd.read_csv(r'C:\Users\bodhe\Downloads\DS\____Projects_____\ticket_pricing_project\data\cleaned\cleaned_competitor_prices.csv')
social = pd.read_csv(r'C:\Users\bodhe\Downloads\DS\____Projects_____\ticket_pricing_project\data\cleaned\cleaned_social_media_buzz.csv')

In [13]:
# converting date columns to datetime format
tickets['event_date'] = pd.to_datetime(tickets['event_date'])
tickets['sale_date'] = pd.to_datetime(tickets['sale_date'])
weather['date'] = pd.to_datetime(weather['date'])
social['date'] = pd.to_datetime(social['date'])

In [26]:
# Time Features  
tickets['event_month'] = tickets['event_date'].dt.month
tickets['event_day_of_week'] = tickets['event_date'].dt.dayofweek
tickets['event_quarter'] = tickets['event_date'].dt.quarter
tickets['is_weekend'] = tickets['event_date'].dt.dayofweek.isin([5, 6]).astype(int)
tickets['days_until_event'] = (tickets['event_date'] - tickets['sale_date']).dt.days

In [39]:
# lower the city names in weather data for easier merging
weather['city'] = weather['city'].str.lower()
tickets['city'] = tickets['city'].str.lower()

In [40]:
# Weather features 
tickets = tickets.merge(weather, left_on=['event_date', 'city'], right_on=['date', 'city'], how='left')

if 'date' in tickets.columns:
    tickets.drop(columns=['date'], inplace=True)

In [45]:
tickets.columns

Index(['ticket_id', 'event_id', 'event_name', 'event_type', 'event_date',
       'sale_date', 'days_until_event', 'seat_section', 'ticket_price',
       'quantity_sold', 'city', 'venue_capacity', 'revenue', 'event_month',
       'is_weekend', 'event_quarter', 'event_day_of_week', 'temperature_x',
       'rain_x', 'humidity_x', 'temperature_y', 'rain_y', 'humidity_y',
       'temperature', 'rain', 'humidity'],
      dtype='object')

In [53]:
tickets['temperature'].fillna(tickets['temperature'].mean(), inplace=True)
tickets['rain'].fillna(0, inplace=True)
tickets['humidity'].fillna(tickets['humidity'].mean(), inplace=True)


In [ ]:
# competitor features

comp_avg = competitors.groupby(['event_id', 'seat_section'])['competitor_price'].mean().reset_index()
comp_avg.columns = ['event_id', 'seat_section', 'avg_competitor_price']

tickets = tickets.merge(comp_avg, on=['event_id', 'seat_section'], how='left')

In [65]:
# Social media features

social_avg = social.groupby('event_id').agg({
    'mentions': 'mean',
    'sentiment_score': 'mean',
    'trending' :'max'
}).reset_index()
social_avg.columns = ['event_id', 'avg_mentions', 'avg_sentiment_score', 'is_trending']

tickets = tickets.merge(social_avg, on='event_id', how='left')
tickets['avg_mentions'].fillna(0, inplace=True)
tickets['avg_sentiment_score'].fillna(0, inplace=True)
tickets['is_trending'].fillna(0, inplace=True)

In [69]:
# Encoding categorical variables

tickets = pd.get_dummies(tickets, columns=['event_type', 'seat_section' , 'city'], dtype=int)

In [70]:
final_file = r'C:\Users\bodhe\Downloads\DS\____Projects_____\ticket_pricing_project\data\cleaned\feature_engineered_ticket_data.csv'
tickets.to_csv(final_file, index=False)

In [71]:
new = pd.read_csv(r'C:\Users\bodhe\Downloads\DS\____Projects_____\ticket_pricing_project\data\cleaned\feature_engineered_ticket_data.csv')
new.head()

,ticket_id,event_id,event_name,event_date,sale_date,days_until_event,ticket_price,quantity_sold,venue_capacity,revenue,...,seat_section_floor,seat_section_general admission,seat_section_lower bowl,seat_section_upper bowl,seat_section_vip,city_chicago,city_los angeles,city_miami,city_new york,city_seattle
0,TKT_000001,EVT_0000,Taylor Swift - Chicago,2024-05-05,2024-02-10,85,330.04,2,5000,660.08,...,0,0,0,0,1,1,0,0,0,0
1,TKT_000002,EVT_0000,Taylor Swift - Chicago,2024-05-05,2024-03-06,60,235.89,1,5000,235.89,...,1,0,0,0,0,1,0,0,0,0
2,TKT_000003,EVT_0000,Taylor Swift - Chicago,2024-05-05,2024-04-07,28,118.07,3,5000,354.21,...,0,0,0,1,0,1,0,0,0,0
3,TKT_000004,EVT_0000,Taylor Swift - Chicago,2024-05-05,2024-03-17,49,194.82,3,5000,584.46,...,0,0,1,0,0,1,0,0,0,0
4,TKT_000005,EVT_0000,Taylor Swift - Chicago,2024-05-05,2024-04-02,33,422.28,3,5000,1266.84,...,0,0,0,0,1,1,0,0,0,0
